# rearrange-as-sequential-layer — worked example 2: ViT-style patchify with a Rearrange layer

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `rearrange-as-sequential-layer`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
from einops.layers.torch import Rearrange

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The ViT patch-embedding step is a single `Rearrange('b c (h ph) (w pw) -> b (h w) (ph pw c)')` layer that splits an image into non-overlapping patches and flattens each, followed by a Linear projection. Placing it in `nn.Sequential` keeps the patch logic declarative.

## Worked solution

We build `patchify_sequential(in_channels, height, width, patch_size, embed_dim)`. The `Rearrange` pattern groups each spatial axis into a patch-count times patch-size factor, then re-collects the patch grid into a sequence axis `(h w)` and flattens each patch's pixels into `(ph pw c)`. The resulting `(B, num_patches, patch_dim)` tensor is projected by a `Linear(patch_dim, embed_dim)`. We compute `patch_dim = patch_size*patch_size*in_channels`, build the Sequential, and run a batch through it. We print the output shape, confirming `(B, num_patches, embed_dim)` where `num_patches = (H//p)*(W//p)`.

In [ ]:
import torch as t
from einops.layers.torch import Rearrange

t.manual_seed(1)

def patchify_sequential(in_channels, height, width, patch_size, embed_dim):
    patch_dim = patch_size * patch_size * in_channels
    return t.nn.Sequential(
        Rearrange('b c (h ph) (w pw) -> b (h w) (ph pw c)',
                  ph=patch_size, pw=patch_size),
        t.nn.Linear(patch_dim, embed_dim),
    )

model = patchify_sequential(3, 16, 16, 4, 32)
out = model(t.randn(2, 3, 16, 16))
print('output shape:', tuple(out.shape))  # (2, 16, 32)